# Customer Support Router Workflow

This workflow builds a **customer support triage system** that classifies
incoming requests and routes them to the right specialist agent. The pattern
is **triage → (human review when uncertain) → route → resolve**:

1. A **Triage Agent** analyzes the customer message and returns a structured
   JSON classification (category, urgency, confidence, summary).
2. The workflow's **ConditionGroup** inspects the classification. When triage is
   confident, it routes straight to the right specialist.
3. When triage is **not** confident, the workflow does not guess. It **pauses**
   with a `Question` node and asks a human reviewer to confirm the category or
   escalate — a true human-in-the-loop gate. The workflow resumes and routes on
   the human's decision.
4. The specialist agent handles the issue and the customer receives a response.

## Step 1: Install Packages

In [ ]:
%pip install python-dotenv azure-ai-projects==2.2.0 azure-identity

## Step 2: Configure Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)
load_dotenv(override=True)

project_url = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

print("Project URL:", project_url)
print("Deployment:", deployment_name)

## Step 3: Create the Foundry Client & Helpers

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, WorkflowAgentDefinition
from azure.identity import DefaultAzureCredential

foundry_client = AIProjectClient(
    endpoint=project_url,
    credential=DefaultAzureCredential(),
    allow_preview=True,  # workflows are a preview feature; required to register WorkflowAgentDefinition
)

oai = foundry_client.get_openai_client()


def register_agent(name: str, instructions: str, description: str = ""):
    """Register a prompt agent in Foundry."""
    result = foundry_client.agents.create_version(
        agent_name=name,
        definition=PromptAgentDefinition(
            model=deployment_name,
            instructions=instructions,
        ),
        description=description,
    )
    print(f"Registered agent: {result.name} (version {result.version})")
    return result


def register_workflow(name: str, yaml_str: str, description: str = ""):
    """Register a workflow in Foundry."""
    result = foundry_client.agents.create_version(
        agent_name=name,
        definition=WorkflowAgentDefinition(workflow=yaml_str),
        description=description,
    )
    print(f"Registered workflow: {result.name} (version {result.version})")
    return result


def invoke_agent(agent_name: str, user_input: str):
    """Create a conversation and invoke a registered agent."""
    session = oai.conversations.create()
    response = oai.responses.create(
        conversation=session.id,
        input=user_input,
        extra_body={"agent_reference": {"name": agent_name, "type": "agent_reference"}},
    )
    return response.output_text


print("Foundry client ready.")

## Step 4: Register the Agents

We register four agents, each with a specific role:

1. **triage-agent** -- Analyzes the customer message and returns a JSON
   classification with `category`, `urgency`, `confidence`, and `summary`.
2. **billing-specialist** -- Handles billing, payment, and refund issues.
3. **technical-support** -- Handles bugs, errors, and technical difficulties.
   Also serves as the default route for `account` and `general` categories.
4. **human-handoff** -- Acknowledges the request and generates a reference
   number. This is the escalation path: when triage confidence is low, a human
   reviewer is asked to decide, and `human-handoff` is where the reviewer sends
   anything that needs a person rather than an automated specialist.

In [ ]:
# --- Triage Agent ---
register_agent(
    name="triage-agent",
    instructions="""You are a customer support triage specialist. Your job is to analyze incoming customer support requests and classify them accurately.

Analyze the customer's message and return ONLY a JSON object with these fields:

1. "category": Classify into exactly one of these categories:
   - "billing" -- payment issues, charges, invoices, refunds, subscription changes, pricing questions
   - "technical" -- bugs, errors, crashes, performance problems, feature not working, integration issues
   - "account" -- login problems, password resets, account settings, profile changes, permissions
   - "general" -- product questions, feedback, feature requests, anything that does not fit the above

2. "urgency": Rate from 1 to 5:
   - 1 = informational / no rush
   - 2 = low priority, can wait a few days
   - 3 = normal priority
   - 4 = high priority, affecting work
   - 5 = critical, service down or money lost

3. "confidence": A decimal between 0.0 and 1.0 representing how confident you are in your classification. Use lower confidence (below 0.7) when the message is vague, spans multiple categories, or lacks enough information.

4. "summary": A one-sentence summary of the customer's issue.

Return ONLY valid JSON. No markdown, no code fences, no extra text.""",
    description="Classifies support requests into category/urgency/confidence/summary as JSON.",
)

In [ ]:
# --- Billing Specialist ---
register_agent(
    name="billing-specialist",
    instructions="""You are a billing support specialist. You handle all payment, invoice, subscription, and refund-related inquiries.

When responding to customers:
- Be empathetic and professional
- Reference specific billing policies when applicable
- If a refund is requested, explain the refund process and timeline
- If a charge is disputed, walk through how to verify the charge
- For subscription changes, outline the available plans and how to switch
- Always end with a clear next step or resolution

Address the customer's billing concern directly.""",
    description="Handles billing, payment, and refund inquiries.",
)

# --- Technical Support ---
register_agent(
    name="technical-support",
    instructions="""You are a technical support specialist. You help customers troubleshoot software bugs, errors, performance issues, integration problems, and other technical difficulties.

When responding to customers:
- Ask clarifying questions if the issue is underspecified
- Provide step-by-step troubleshooting instructions
- Suggest workarounds when a permanent fix is not immediately available
- Reference relevant documentation or knowledge base articles when applicable
- If the issue requires escalation to engineering, clearly state that and set expectations

Address the customer's technical concern directly.""",
    description="Handles bugs, errors, and technical issues.",
)

# --- Human Handoff ---
register_agent(
    name="human-handoff",
    instructions="""You are a customer service representative. The system was unable to confidently classify the customer's request, so it is being forwarded to a human agent.

Your job is to:
1. Acknowledge the customer's message
2. Let them know their request has been forwarded to a human agent for review
3. Provide a reference number in the format SR-XXXXX (generate a random 5-digit number)
4. Reassure them that someone will follow up within 24 hours

Be warm, professional, and concise.""",
    description="Handles low-confidence triage results with human handoff.",
)

## Step 5: Define and Register the Workflow

The workflow is defined in **CSDL YAML**. Here is how it works:

1. **SetVariable** -- Captures the user's message.
2. **InvokeAzureAgent** (triage-agent) -- Classifies the request. The
   `responseObject` output parses the agent's JSON into `Local.TriageResult`.
3. **ConditionGroup** -- Branches explicitly on the parsed fields. Order matters — the first matching condition wins:
   - `confidence <= 0.7` → **human-in-the-loop gate**. Instead of guessing on a shaky classification, the workflow runs a `Question` node that **pauses execution and asks a human reviewer to decide**. The reviewer's reply is saved to `Local.ReviewerDecision`, and a nested `ConditionGroup` routes on it: `billing` → `billing-specialist`; `technical` / `account` / `general` → `technical-support`; anything else (including `human`) → `human-handoff`. The `<=` (not `<`) means a tie at exactly `0.7` is treated as low-confidence and goes to review. Tune this boundary for your own risk tolerance.
   - `category == "billing"` → `billing-specialist`.
   - `category == "technical"` → `technical-support`.
   - `category == "account"` → `technical-support`. Account issues (login, password reset, permissions) overlap heavily with tech support, so we reuse that specialist.
   - `category == "general"` → `technical-support`. General questions also route to the tech team, who fields product and feature inquiries.
   - Default (`true`) → `human-handoff`. An unknown or unrecognized category with high confidence is rare; treat it as an escalation.
4. **EndConversation** -- Closes the session.

> **Human-in-the-loop design choice:** The `Question` node turns a one-shot
> automated run into a multi-turn conversation. On a low-confidence request the
> workflow does not complete — it emits the reviewer question and **suspends**.
> The next message on the same conversation supplies the human's decision and
> resumes the workflow from where it paused. High-confidence requests never hit
> the gate and complete in a single turn. This is the difference between a system
> that *acts on* an uncertain guess and one that *asks a person first*.

> **Triage context design choice:** The triage agent receives only the latest
> user message (`Local.LatestMessage`). Multi-turn triage — passing the full
> conversation history — is a valid variant, but it increases triage token
> cost and makes the classification sensitive to earlier turns. Start with
> single-message triage and add history only when your routing accuracy
> suffers.

In [ ]:
WORKFLOW_YAML = """
kind: workflow
name: customer-support-router
description: Triage and route customer support requests, pausing for a human reviewer when triage confidence is low.
trigger:
  kind: OnConversationStart
  id: trigger_start
  actions:
    - kind: SetVariable
      id: init_msg
      variable: Local.LatestMessage
      value: =UserMessage(System.LastMessageText)

    - kind: InvokeAzureAgent
      id: invoke_triage
      agent:
        name: triage-agent
        conversationId: =System.ConversationId
      input:
        messages: =Local.LatestMessage
      output:
        responseObject: Local.TriageResult

    - kind: ConditionGroup
      id: route_request
      conditions:
        - id: route_low_confidence
          condition: =Local.TriageResult.confidence <= 0.7
          actions:
            # --- Human-in-the-loop gate ---
            # Triage is unsure. Instead of acting on a shaky guess, the workflow
            # pauses here and asks a human reviewer to decide. Execution suspends
            # until the next message on this conversation supplies the answer.
            - kind: Question
              id: ask_reviewer
              interruptionPolicy:
                allowInterruption: true
              prompt: "Triage was not confident about this request (it suggested '{Local.TriageResult.category}'). Reply with the correct category to route it -- billing, technical, account, or general -- or reply 'human' to hand it to a person."
              variable: init:Local.ReviewerDecision
              entity: StringPrebuiltEntity

            # The workflow resumes here once the reviewer answers. Route on their
            # decision. Lower/Trim make the match tolerant of casing and spaces.
            - kind: ConditionGroup
              id: route_reviewer_decision
              conditions:
                - id: reviewer_billing
                  condition: =Lower(Trim(Local.ReviewerDecision)) = "billing"
                  actions:
                    - kind: InvokeAzureAgent
                      id: invoke_billing_reviewed
                      agent:
                        name: billing-specialist
                        conversationId: =System.ConversationId
                      input:
                        messages: =Local.LatestMessage
                      output:
                        messages: Local.LatestMessage
                      autoSend: true

                - id: reviewer_technical
                  condition: =Lower(Trim(Local.ReviewerDecision)) = "technical"
                  actions:
                    - kind: InvokeAzureAgent
                      id: invoke_technical_reviewed
                      agent:
                        name: technical-support
                        conversationId: =System.ConversationId
                      input:
                        messages: =Local.LatestMessage
                      output:
                        messages: Local.LatestMessage
                      autoSend: true

                - id: reviewer_account
                  condition: =Lower(Trim(Local.ReviewerDecision)) = "account"
                  actions:
                    - kind: InvokeAzureAgent
                      id: invoke_account_reviewed
                      agent:
                        name: technical-support
                        conversationId: =System.ConversationId
                      input:
                        messages: =Local.LatestMessage
                      output:
                        messages: Local.LatestMessage
                      autoSend: true

                - id: reviewer_general
                  condition: =Lower(Trim(Local.ReviewerDecision)) = "general"
                  actions:
                    - kind: InvokeAzureAgent
                      id: invoke_general_reviewed
                      agent:
                        name: technical-support
                        conversationId: =System.ConversationId
                      input:
                        messages: =Local.LatestMessage
                      output:
                        messages: Local.LatestMessage
                      autoSend: true

                - id: reviewer_escalate
                  condition: =true
                  actions:
                    - kind: InvokeAzureAgent
                      id: invoke_handoff_reviewed
                      agent:
                        name: human-handoff
                        conversationId: =System.ConversationId
                      input:
                        messages: =Local.LatestMessage
                      output:
                        messages: Local.LatestMessage
                      autoSend: true

        - id: route_billing
          condition: =Local.TriageResult.category = "billing"
          actions:
            - kind: InvokeAzureAgent
              id: invoke_billing
              agent:
                name: billing-specialist
                conversationId: =System.ConversationId
              input:
                messages: =Local.LatestMessage
              output:
                messages: Local.LatestMessage
              autoSend: true

        - id: route_technical
          condition: =Local.TriageResult.category = "technical"
          actions:
            - kind: InvokeAzureAgent
              id: invoke_technical
              agent:
                name: technical-support
                conversationId: =System.ConversationId
              input:
                messages: =Local.LatestMessage
              output:
                messages: Local.LatestMessage
              autoSend: true

        - id: route_account
          condition: =Local.TriageResult.category = "account"
          actions:
            - kind: InvokeAzureAgent
              id: invoke_account
              agent:
                name: technical-support
                conversationId: =System.ConversationId
              input:
                messages: =Local.LatestMessage
              output:
                messages: Local.LatestMessage
              autoSend: true

        - id: route_general
          condition: =Local.TriageResult.category = "general"
          actions:
            - kind: InvokeAzureAgent
              id: invoke_general
              agent:
                name: technical-support
                conversationId: =System.ConversationId
              input:
                messages: =Local.LatestMessage
              output:
                messages: Local.LatestMessage
              autoSend: true

        - id: route_unknown
          condition: =true
          actions:
            - kind: InvokeAzureAgent
              id: invoke_handoff_unknown
              agent:
                name: human-handoff
                conversationId: =System.ConversationId
              input:
                messages: =Local.LatestMessage
              output:
                messages: Local.LatestMessage
              autoSend: true

    - kind: EndConversation
      id: end_conversation
""".strip()

register_workflow(
    name="customer-support-router",
    yaml_str=WORKFLOW_YAML,
    description="Triage-route-resolve customer support with a human-in-the-loop gate on low-confidence requests.",
)

## Step 6: Verify the Workflow Executes

Let's confirm the workflow runs end-to-end. We check that all actions
complete and inspect which branch the `ConditionGroup` took.

The third test exercises the **human-in-the-loop gate**. A vague request makes
triage unsure, so the workflow does not finish on the first turn — it asks the
reviewer a question and suspends. We then send a second message **on the same
conversation** with the reviewer's decision, and the workflow resumes and routes
on it. Notice that high-confidence requests (Tests 1 and 2) finish in one turn
and never reach the gate.

> **Note:** In the current SDK preview, the workflow's agent responses are
> delivered inline (`autoSend`) but are not included in the Python SDK's
> `output_text` field. To see the full rendered responses — including the
> reviewer question rendered interactively — use the portal's built-in test
> pane (Step 8) or invoke agents directly (Step 7).

In [ ]:
def run_turn(conversation_id, text, label):
    """Send one message to the workflow on a conversation and print the action trace."""
    response = oai.responses.create(
        conversation=conversation_id,
        input=text,
        extra_body={"agent_reference": {"name": "customer-support-router", "type": "agent_reference"}},
    )
    print(f"--- {label} ---")
    print(f"Input: {text!r}")
    try:
        for action in response.output:
            print(f"  Action: {action.action_id} | Kind: {action.kind} | Status: {action.status}")
    except AttributeError:
        # Response format may vary — print raw output if trace attributes are not available
        for item in response.output:
            print(f"  {item}")
    if getattr(response, "output_text", None):
        print(f"  Output text: {response.output_text}")
    print()
    return response


# Test 1: Clear billing request → routes straight to billing-specialist (single turn)
session = oai.conversations.create()
run_turn(session.id,
         "I was charged $49.99 twice on my last invoice. I need a refund.",
         "High confidence: billing (one turn)")

# Test 2: Clear technical request → routes straight to technical-support (single turn)
session2 = oai.conversations.create()
run_turn(session2.id,
         "The app crashes when I export a PDF. Error: timeout after 30 seconds.",
         "High confidence: technical (one turn)")

# Test 3: Vague request → HUMAN-IN-THE-LOOP gate (two turns on the SAME conversation)
session3 = oai.conversations.create()
# Turn 1: triage is unsure, so the workflow pauses and asks the reviewer.
run_turn(session3.id, "help",
         "Low confidence turn 1: workflow asks the reviewer and suspends")
# Turn 2: the human reviewer answers on the same conversation. The workflow
# resumes from the Question node and routes on the reviewer's decision.
run_turn(session3.id, "billing",
         "Low confidence turn 2: reviewer answers 'billing' → workflow resumes and routes")

## Step 7: Test the Agents

Now let's invoke the agents directly to see their actual output. We simulate
the full triage → route → resolve pipeline using Python.

### Test Case 1: Clear Billing Issue

In [ ]:
import json
import re


def strip_code_fences(text):
    """Strip markdown code fences if the LLM wraps its JSON response."""
    text = re.sub(r'^```(?:json)?\s*\n?', '', text.strip())
    text = re.sub(r'\n?```\s*$', '', text)
    return text


def route_support_request(customer_message: str, reviewer_decision: str = None):
    """Run the triage-route-resolve pipeline via direct agent invocation.

    On low confidence this mirrors the workflow's human-in-the-loop gate: a human
    reviewer decides the category. Pass `reviewer_decision` to simulate that input
    (in the live workflow the reviewer types it into the conversation). To make it
    truly interactive in a notebook, replace it with `input("Reviewer decision: ")`.
    """

    # Step 1: Triage
    triage_json = invoke_agent("triage-agent", customer_message)
    triage = json.loads(strip_code_fences(triage_json))
    print(f"Triage: category={triage['category']}, urgency={triage['urgency']}, "
          f"confidence={triage['confidence']}")
    print(f"Summary: {triage['summary']}\n")

    # Step 2: Route. High confidence routes automatically; low confidence pauses
    # for a human reviewer — the human-in-the-loop gate.
    if triage["confidence"] <= 0.7:
        decision = (reviewer_decision or "human").strip().lower()
        print(f"--- LOW CONFIDENCE: human reviewer decides -> {decision!r} ---")
        if decision == "billing":
            response = invoke_agent("billing-specialist", customer_message)
        elif decision in ("technical", "account", "general"):
            response = invoke_agent("technical-support", customer_message)
        else:
            response = invoke_agent("human-handoff", customer_message)
    elif triage["category"] == "billing":
        print("--- Routing to BILLING SPECIALIST ---")
        response = invoke_agent("billing-specialist", customer_message)
    else:
        print(f"--- Routing to TECHNICAL SUPPORT (category: {triage['category']}) ---")
        response = invoke_agent("technical-support", customer_message)

    print(response)


route_support_request(
    "I was charged $49.99 twice on my last invoice dated January 15th. "
    "I only have one subscription and should have been charged once. "
    "I need a refund for the duplicate charge."
)

### Test Case 2: Clear Technical Issue

In [ ]:
route_support_request(
    "The application crashes every time I try to export a PDF report. I get an error "
    "message that says 'Export failed: timeout after 30 seconds.' This started happening "
    "after the latest update. I am using version 3.2.1 on Windows 11."
)

### Test Case 3: Ambiguous Issue (Low Confidence → Human Reviewer Decides)

Triage is unsure, so the human-in-the-loop gate fires. Here we simulate the
reviewer deciding the request is a billing matter by passing
`reviewer_decision="billing"`. Change it to `"human"` (or omit it) to send the
request to `human-handoff` instead.

In [ ]:
route_support_request("help", reviewer_decision="billing")

### Test Case 4: Account-Related Issue

In [ ]:
route_support_request(
    "I cannot log into my account. I have tried resetting my password three times "
    "but I never receive the reset email. My email address is correct and I have "
    "checked spam. My username is jsmith_2024."
)

### Test Case 5: General Inquiry

In [ ]:
route_support_request(
    "I am evaluating your product for my team of 50 people. Can you tell me about "
    "your enterprise plan features, specifically around SSO integration and admin "
    "controls? Also, is there a free trial available?"
)